# American Prompt Evaluation
## Japanese vs American 4-Step Pipeline Comparison

Compare 8 metrics across 3 models (GPT-5, GPT-4o temp=0, GPT-4o temp=1):
- Step 1.1: F1_Participants, F1_Restaurants, Final_Restaurant_Match
- Step 1.2: F1_Suggestion, F1_Response
- Step 2: F1_Mentioned
- Step 3: F1_PerceptionTable
- Step 4: F1_InterpretationTable (Positive-F1)

In [1]:
import json
import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict

## Configuration

In [2]:
DATA_DIR = 'data'
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

# Japanese 4-step results (original)
JAPANESE_DIRS = {
    'GPT-5':     'results/gpt5/raw',
    'GPT-4o': 'results/gpt4o/raw',
}

# American 4-step results
AMERICAN_DIRS = {
    'GPT-5':     'appendix/results/american/gpt5/raw',
    'GPT-4o': 'appendix/results/american/gpt4o/raw',
}

METRIC_NAMES = ['F1_Participants', 'F1_Restaurants', 'Final_Restaurant_Match',
                'F1_Suggestion', 'F1_Response', 'F1_Mentioned',
                'F1_PerceptionTable', 'F1_InterpretationTable']

print('Configuration loaded')
print(f'Gold dir: {GOLD_DIR}')
for label in JAPANESE_DIRS:
    j_exists = os.path.isdir(JAPANESE_DIRS[label])
    a_exists = os.path.isdir(AMERICAN_DIRS[label])
    print(f'  {label}: Japanese={j_exists}, American={a_exists}')

Configuration loaded
Gold dir: Data/gold
  GPT-5: Japanese=True, American=True
  GPT-4o: Japanese=True, American=True


## Evaluation Functions

In [3]:
def normalize_name(x):
    return str(x).strip().lower()

def f1_from_counts(tp, fp, fn):
    if tp == 0:
        return 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def set_f1(gold_list, pred_list):
    if not gold_list and not pred_list:
        return 1.0
    if not gold_list or not pred_list:
        return 0.0
    gold_set = {str(x).strip() for x in gold_list}
    pred_set = {str(x).strip() for x in pred_list}
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    return f1_from_counts(tp, fp, fn)

def table_f1(gold_table, pred_table, key_columns):
    if not gold_table and not pred_table:
        return 1.0
    if not gold_table or not pred_table:
        return 0.0
    def make_key(row):
        return ' '.join(str(row.get(c, '')).strip() for c in key_columns)
    gold_keys = set(make_key(r) for r in gold_table)
    pred_keys = set(make_key(r) for r in pred_table)
    tp = len(gold_keys & pred_keys)
    fp = len(pred_keys - gold_keys)
    fn = len(gold_keys - pred_keys)
    return f1_from_counts(tp, fp, fn)

def mentioned_f1(gold_table, pred_table, scope_restaurants):
    scope_r = {normalize_name(r) for r in scope_restaurants}
    def extract_mentioned_pairs(table):
        pairs = set()
        for row in table:
            if row.get('mention', '') == 'Mentioned':
                r = normalize_name(row.get('restaurant', ''))
                if r in scope_r:
                    pairs.add((row['participant'], r))
        return pairs
    gold_pairs = extract_mentioned_pairs(gold_table)
    pred_pairs = extract_mentioned_pairs(pred_table)
    if not gold_pairs and not pred_pairs:
        return 1.0
    tp = len(gold_pairs & pred_pairs)
    fp = len(pred_pairs - gold_pairs)
    fn = len(gold_pairs - pred_pairs)
    return f1_from_counts(tp, fp, fn)

def perception_micro_f1(gold_table, pred_table, scope_p, scope_r):
    scope_r_lower = {normalize_name(r) for r in scope_r}
    gold_map = {}
    for g in gold_table:
        if g['participant'] in scope_p and normalize_name(g['restaurant']) in scope_r_lower:
            key = (g['participant'], normalize_name(g['restaurant']))
            gold_map[key] = g.get('perception', g.get('sentiment', '')).strip()
    pred_map = {}
    for p in pred_table:
        if p['participant'] in scope_p and normalize_name(p['restaurant']) in scope_r_lower:
            key = (p['participant'], normalize_name(p['restaurant']))
            pred_map[key] = p.get('sentiment', p.get('perception', '')).strip()
    tp = fp = fn = 0
    for key, gv in gold_map.items():
        if key in pred_map:
            if gv == pred_map[key]:
                tp += 1
            else:
                fn += 1
        else:
            fn += 1
    for key in pred_map:
        if key not in gold_map:
            fp += 1
    return f1_from_counts(tp, fp, fn)

def parse_factors(s):
    if not s or str(s).strip().lower() == 'none':
        return set()
    return {f.strip() for f in str(s).split(',') if f.strip() and f.strip().lower() != 'none'}

def positive_f1_cell(gold_set, pred_set):
    inter = len(gold_set & pred_set)
    prec = inter / len(pred_set) if pred_set else 0.0
    rec  = inter / len(gold_set) if gold_set else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

def integrate_tables(pref_tbl, cons_tbl):
    union_map = {}
    for row in pref_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('preferences', 'None')))
    for row in cons_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('constraints', 'None')))
    return [
        {'participant': p, 'restaurant': r,
         'integrated': ','.join(sorted(factors)) if factors else 'None'}
        for (p, r), factors in union_map.items()
    ]

def interpretation_positive_f1(gold_pref, gold_cons, pred_pref, pred_cons, scope_p, scope_r):
    scope_r_lower = {normalize_name(r) for r in scope_r}
    def filt(table):
        return [t for t in table
                if t['participant'] in scope_p
                and normalize_name(t['restaurant']) in scope_r_lower]
    gp, gc = filt(gold_pref), filt(gold_cons)
    pp, pc = filt(pred_pref), filt(pred_cons)
    gold_int = integrate_tables(gp, gc)
    pred_int = integrate_tables(pp, pc)
    gold_map = {(it['participant'], normalize_name(it['restaurant'])): parse_factors(it['integrated']) for it in gold_int}
    pred_map = {(it['participant'], normalize_name(it['restaurant'])): parse_factors(it['integrated']) for it in pred_int}
    pos_sum = pos_cnt = 0
    for key, gold_set in gold_map.items():
        pred_set = pred_map.get(key, set())
        if gold_set:
            pos_sum += positive_f1_cell(gold_set, pred_set)
            pos_cnt += 1
    return pos_sum / pos_cnt if pos_cnt else 0.0

print('Evaluation functions loaded')

Evaluation functions loaded


## Data Loading

In [4]:
def load_gold(log_name):
    gold_path = os.path.join(GOLD_DIR, log_name)
    gold = {}
    for key, fname in [('step1_1', 'step1_1_gold.json'), ('step1_2', 'step1_2_gold.json'),
                       ('step2', 'step2_gold.json'), ('step3', 'step3_gold.json'),
                       ('step4', 'step4_gold.json')]:
        fpath = os.path.join(gold_path, fname)
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                gold[key] = json.load(f)
    return gold

def load_4step_final(result_dir, log_name):
    fpath = os.path.join(result_dir, log_name, f'{log_name}_final_results.json')
    if not os.path.exists(fpath):
        return None
    with open(fpath, 'r', encoding='utf-8') as f:
        return json.load(f)

def evaluate_all_metrics(pred, gold, scope_p, scope_r):
    metrics = {}
    gold_s11 = gold.get('step1_1', {})
    metrics['F1_Participants'] = set_f1(gold_s11.get('participants', []), pred.get('participants', []))
    metrics['F1_Restaurants'] = set_f1(gold_s11.get('restaurant_brands', []), pred.get('restaurant_brands', []))
    metrics['Final_Restaurant_Match'] = (
        1.0 if gold_s11.get('final_restaurant', '').strip() == pred.get('final_restaurant', '').strip()
        else 0.0
    )
    gold_s12 = gold.get('step1_2', {})
    metrics['F1_Suggestion'] = table_f1(gold_s12.get('suggestion_table', []), pred.get('suggestion_table', []),
                                        ['participant', 'suggestion_type'])
    metrics['F1_Response'] = table_f1(gold_s12.get('response_table', []), pred.get('response_table', []),
                                      ['participant', 'response_type'])
    gold_s2 = gold.get('step2', {})
    metrics['F1_Mentioned'] = mentioned_f1(gold_s2.get('mentioned_table', []),
                                            pred.get('mentioned_table', []), scope_r)
    gold_s3 = gold.get('step3', {})
    gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
    pred_s3_table = pred.get('sentiment_table', pred.get('perception_table', []))
    metrics['F1_PerceptionTable'] = perception_micro_f1(gold_s3_table, pred_s3_table, scope_p, scope_r)
    gold_s4 = gold.get('step4', {})
    metrics['F1_InterpretationTable'] = interpretation_positive_f1(
        gold_s4.get('preference_table', []), gold_s4.get('constraint_table', []),
        pred.get('preference_table', []), pred.get('constraint_table', []),
        scope_p, scope_r
    )
    return metrics

print('Data loading & evaluation functions loaded')

Data loading & evaluation functions loaded


## Run Evaluation: Japanese vs American

In [5]:
all_rows = []

for persona, dirs in [('Japanese', JAPANESE_DIRS), ('American', AMERICAN_DIRS)]:
    for model_label in dirs:
        result_dir = dirs[model_label]
        if not os.path.isdir(result_dir):
            print(f'  SKIP {persona}/{model_label}: {result_dir} not found')
            continue

        logs = sorted([d for d in os.listdir(result_dir)
                       if d.endswith('_log') and os.path.isdir(os.path.join(result_dir, d))])

        completed = 0
        skipped = 0
        for log_name in logs:
            gold = load_gold(log_name)
            if not gold:
                skipped += 1
                continue
            final = load_4step_final(result_dir, log_name)
            if not final:
                skipped += 1
                continue

            scope_p = set(final['step1']['participants'])
            scope_r = set(final['step1']['restaurant_brands'])

            pred = {
                'participants': final['step1']['participants'],
                'restaurant_brands': final['step1']['restaurant_brands'],
                'final_restaurant': final['step1']['final_restaurant'],
                'suggestion_table': final['step1']['suggestion_table'],
                'response_table': final['step1']['response_table'],
                'mentioned_table': final['step2']['mentioned_table'],
                'sentiment_table': final['step3']['sentiment_table'],
                'preference_table': final['step4']['f1']['preference_table'],
                'constraint_table': final['step4']['f1']['constraint_table'],
            }

            m = evaluate_all_metrics(pred, gold, scope_p, scope_r)
            row = {'persona': persona, 'model': model_label, 'log': log_name}
            row.update(m)
            all_rows.append(row)
            completed += 1

        print(f'  {persona}/{model_label}: {completed} evaluated, {skipped} skipped')

df = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df)}')
df.head()

  Japanese/GPT-5: 47 evaluated, 0 skipped
  Japanese/GPT-4o: 47 evaluated, 0 skipped
  American/GPT-5: 47 evaluated, 0 skipped
  American/GPT-4o: 47 evaluated, 0 skipped

Total rows: 281


,persona,model,log,F1_Participants,F1_Restaurants,Final_Restaurant_Match,F1_Suggestion,F1_Response,F1_Mentioned,F1_PerceptionTable,F1_InterpretationTable
0,Japanese,GPT-5,10_log,1.0,1.0,1.0,0.666667,0.666667,1.0,0.800000,0.533333
1,Japanese,GPT-5,11_log,1.0,1.0,1.0,1.000000,0.400000,1.0,0.903226,0.151852
2,Japanese,GPT-5,12_log,1.0,1.0,1.0,1.000000,1.000000,1.0,0.875000,0.800000
3,Japanese,GPT-5,13_log,1.0,1.0,1.0,1.000000,0.666667,1.0,0.800000,0.375000
4,Japanese,GPT-5,14_log,1.0,1.0,1.0,1.000000,0.000000,1.0,0.923077,0.815152


## Summary: Japanese vs American Comparison

In [6]:
# Model × Persona summary (mean ± std)
summary = df.groupby(['persona', 'model'])[METRIC_NAMES].agg(['mean', 'std']).round(4)
display(summary)

F1_Participants      F1_Restaurants          \
                              mean  std           mean     std   
persona  model                                                   
American GPT-4o             1.0  0.0         0.9687  0.0959   
         GPT-5                 1.0  0.0         1.0000  0.0000   
Japanese GPT-4o             1.0  0.0         0.9884  0.0403   
         GPT-5                 1.0  0.0         0.9981  0.0133   

                   Final_Restaurant_Match         F1_Suggestion          \
                                     mean     std          mean     std   
persona  model                                                            
American GPT-4o                 0.9362  0.2471        0.7518  0.2034   
         GPT-5                     1.0000  0.0000        0.8209  0.2281   
Japanese GPT-4o                 0.9574  0.2040        0.7411  0.1980   
         GPT-5                     0.9787  0.1459        0.8323  0.1907   

                   F1_Response         F1_Mentioned          \
                          mean     std         mean     std   
persona  model                                                
American GPT-4o      0.3447  0.3457       0.9466  0.1584   
         GPT-5          0.5387  0.2941       0.9856  0.0494   
Japanese GPT-4o      0.3365  0.3443       0.9615  0.1197   
         GPT-5          0.5369  0.2907       0.9804  0.0639   

                   F1_PerceptionTable         F1_InterpretationTable          
                                 mean     std                   mean     std  
persona  model                                                                
American GPT-4o             0.8518  0.1379                 0.3651  0.2573  
         GPT-5                 0.8823  0.0904                 0.4501  0.2303  
Japanese GPT-4o             0.8359  0.1482                 0.3675  0.2577  
         GPT-5                 0.8834  0.0923                 0.4340  0.2484

## Side-by-Side Comparison per Model

In [7]:
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n{"="*110}')
    print(f'  {model}')
    print(f'{"="*110}')
    print(f'  {"Metric":<28} | {"Japanese":>22} | {"American":>22} | {"Δ (Amer-Jpn)":>14}')
    print(f'  {"-"*100}')

    jpn = df[(df['model'] == model) & (df['persona'] == 'Japanese')]
    amer = df[(df['model'] == model) & (df['persona'] == 'American')]

    for m in METRIC_NAMES:
        jm = jpn[m].mean() if len(jpn) > 0 else float('nan')
        js = jpn[m].std() if len(jpn) > 0 else float('nan')
        am = amer[m].mean() if len(amer) > 0 else float('nan')
        a_s = amer[m].std() if len(amer) > 0 else float('nan')
        delta = am - jm
        print(f'  {m:<28} | {jm:.4f} (±{js:.4f}) | {am:.4f} (±{a_s:.4f}) | {delta:>+.4f}')


  GPT-5
  Metric                       |               Japanese |               American |   Δ (Amer-Jpn)
  ----------------------------------------------------------------------------------------------------
  F1_Participants              | 1.0000 (±0.0000) | 1.0000 (±0.0000) | +0.0000
  F1_Restaurants               | 0.9981 (±0.0133) | 1.0000 (±0.0000) | +0.0019
  Final_Restaurant_Match       | 0.9787 (±0.1459) | 1.0000 (±0.0000) | +0.0213
  F1_Suggestion                | 0.8323 (±0.1907) | 0.8209 (±0.2281) | -0.0113
  F1_Response                  | 0.5369 (±0.2907) | 0.5387 (±0.2941) | +0.0018
  F1_Mentioned                 | 0.9804 (±0.0639) | 0.9856 (±0.0494) | +0.0052
  F1_PerceptionTable           | 0.8834 (±0.0923) | 0.8823 (±0.0904) | -0.0011
  F1_InterpretationTable       | 0.4340 (±0.2484) | 0.4501 (±0.2303) | +0.0161

  GPT-4o
  Metric                       |               Japanese |               American |   Δ (Amer-Jpn)
  ------------------------------------------------

## Pivot Table

In [8]:
pivot = df.groupby(['persona', 'model'])[METRIC_NAMES].mean().round(4)
display(pivot)

F1_Participants  F1_Restaurants  Final_Restaurant_Match  \
persona  model                                                                
American GPT-4o              1.0          0.9687                  0.9362   
         GPT-5                  1.0          1.0000                  1.0000   
Japanese GPT-4o              1.0          0.9884                  0.9574   
         GPT-5                  1.0          0.9981                  0.9787   

                    F1_Suggestion  F1_Response  F1_Mentioned  \
persona  model                                                 
American GPT-4o         0.7518       0.3447        0.9466   
         GPT-5             0.8209       0.5387        0.9856   
Japanese GPT-4o         0.7411       0.3365        0.9615   
         GPT-5             0.8323       0.5369        0.9804   

                    F1_PerceptionTable  F1_InterpretationTable  
persona  model                                                  
American GPT-4o              0.8518                  0.3651  
         GPT-5                  0.8823                  0.4501  
Japanese GPT-4o              0.8359                  0.3675  
         GPT-5                  0.8834                  0.4340

## Per-Conversation Comparison (optional detail)

In [9]:
# Merge Japanese & American per conversation to see per-conv deltas
delta_rows = []
for model in ['GPT-5', 'GPT-4o']:
    jpn = df[(df['model'] == model) & (df['persona'] == 'Japanese')].set_index('log')
    amer = df[(df['model'] == model) & (df['persona'] == 'American')].set_index('log')
    common_logs = sorted(set(jpn.index) & set(amer.index))

    for log_name in common_logs:
        row = {'model': model, 'log': log_name}
        for m in METRIC_NAMES:
            jv = jpn.loc[log_name, m] if log_name in jpn.index else float('nan')
            av = amer.loc[log_name, m] if log_name in amer.index else float('nan')
            row[f'{m}_jpn'] = jv
            row[f'{m}_amer'] = av
            row[f'{m}_delta'] = av - jv
        delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows)
print(f'Per-conversation comparison: {len(df_delta)} rows')

# Show average delta per model
for model in ['GPT-5', 'GPT-4o']:
    sub = df_delta[df_delta['model'] == model]
    print(f'\n{model} avg delta (American - Japanese):')
    for m in METRIC_NAMES:
        avg_d = sub[f'{m}_delta'].mean()
        print(f'  {m:<28}: {avg_d:>+.4f}')

Per-conversation comparison: 140 rows

GPT-5 avg delta (American - Japanese):
  F1_Participants             : +0.0000
  F1_Restaurants              : +0.0019
  Final_Restaurant_Match      : +0.0213
  F1_Suggestion               : -0.0113
  F1_Response                 : +0.0018
  F1_Mentioned                : +0.0052
  F1_PerceptionTable          : -0.0011
  F1_InterpretationTable      : +0.0161

GPT-4o avg delta (American - Japanese):
  F1_Participants             : +0.0000
  F1_Restaurants              : -0.0197
  Final_Restaurant_Match      : -0.0213
  F1_Suggestion               : +0.0106
  F1_Response                 : +0.0082
  F1_Mentioned                : -0.0149
  F1_PerceptionTable          : +0.0159
  F1_InterpretationTable      : -0.0024

  F1_Participants             : +0.0000
  F1_Restaurants              : -0.0030
  Final_Restaurant_Match      : +0.0217
  F1_Suggestion               : +0.0283
  F1_Response                 : -0.0185
  F1_Mentioned                : -0.0105


## Save Results

In [10]:
df.to_csv('appendix/results/american/american_eval_detail.csv', index=False)

summary_flat = df.groupby(['persona', 'model'])[METRIC_NAMES].agg(['mean', 'std']).reset_index()
summary_flat.to_csv('appendix/results/american/american_eval_summary.csv', index=False)

df_delta.to_csv('appendix/results/american/american_eval_per_conv_delta.csv', index=False)

print('Saved:')
print('  - appendix/results/american/american_eval_detail.csv (per conversation, both personas)')
print('  - appendix/results/american/american_eval_summary.csv (persona × model summary)')
print('  - appendix/results/american/american_eval_per_conv_delta.csv (per-conversation Japanese vs American delta)')

Saved:
  - american_eval_detail.csv (per conversation, both personas)
  - american_eval_summary.csv (persona × model summary)
  - american_eval_per_conv_delta.csv (per-conversation Japanese vs American delta)


## Per-Element Breakdown Analysis
Detailed F1 evaluation per category within each step:
- Step 1.2: Suggestion type (Strong/Moderate/Weak), Response type (Agreeable/Moderate/Disagreeable)
- Step 3: Perception (Positive/Neutral/Negative/Mix) — scope-filtered
- Step 4: Factor-level (A1–A7) Positive-F1 — scope-filtered

In [11]:
# ============================================================
# Per-Element Evaluation Functions
# ============================================================

def step12_per_type_f1(gold_table, pred_table, type_column, type_value):
    """F1 for a specific type value in suggestion/response table.
    E.g., how well does the model predict 'Strong' suggestions specifically?"""
    gold_set = {r['participant'] for r in gold_table if r.get(type_column) == type_value}
    pred_set = {r['participant'] for r in pred_table if r.get(type_column) == type_value}
    if not gold_set and not pred_set:
        return 1.0, 0, 0, 0  # f1, tp, fp, fn
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    return f1_from_counts(tp, fp, fn), tp, fp, fn

def step3_per_perception_f1(gold_table, pred_table, perception_value, scope_p, scope_r):
    """F1 for a specific perception value in Step 3, scope-filtered.
    E.g., how well does the model identify 'Positive' perceptions?"""
    scope_r_lower = {normalize_name(r) for r in scope_r}
    
    def extract_pairs(table, val_key):
        pairs = set()
        for row in table:
            p = row.get('participant', '')
            r = normalize_name(row.get('restaurant', ''))
            v = row.get(val_key, '').strip()
            if p in scope_p and r in scope_r_lower and v == perception_value:
                pairs.add((p, r))
        return pairs
    
    gold_pairs = extract_pairs(gold_table, 'perception')
    pred_pairs = extract_pairs(pred_table, 'sentiment')  # pred uses 'sentiment'
    # Also try 'perception' key for pred
    if not pred_pairs:
        pred_pairs = extract_pairs(pred_table, 'perception')
    
    if not gold_pairs and not pred_pairs:
        return 1.0, 0, 0, 0
    tp = len(gold_pairs & pred_pairs)
    fp = len(pred_pairs - gold_pairs)
    fn = len(gold_pairs - pred_pairs)
    return f1_from_counts(tp, fp, fn), tp, fp, fn

def step4_per_factor_positive_f1(gold_pref, gold_cons, pred_pref, pred_cons, 
                                   scope_p, scope_r, target_factor):
    """Positive-F1 for a specific factor (e.g., 'A1') in Step 4, scope-filtered.
    For each (participant, restaurant) cell, check if the target factor is in gold and/or pred."""
    scope_r_lower = {normalize_name(r) for r in scope_r}
    
    def filt(table):
        return [t for t in table
                if t['participant'] in scope_p
                and normalize_name(t['restaurant']) in scope_r_lower]
    
    gp, gc = filt(gold_pref), filt(gold_cons)
    pp, pc = filt(pred_pref), filt(pred_cons)
    
    # Build integrated maps
    def build_factor_map(pref_tbl, cons_tbl):
        m = {}
        for row in pref_tbl:
            key = (row['participant'], normalize_name(row['restaurant']))
            m.setdefault(key, set()).update(parse_factors(row.get('preferences', 'None')))
        for row in cons_tbl:
            key = (row['participant'], normalize_name(row['restaurant']))
            m.setdefault(key, set()).update(parse_factors(row.get('constraints', 'None')))
        return m
    
    gold_map = build_factor_map(gp, gc)
    pred_map = build_factor_map(pp, pc)
    
    # Count cells where target_factor appears in gold (positive cells for this factor)
    tp = fp = fn = 0
    for key, gold_factors in gold_map.items():
        g_has = target_factor in gold_factors
        p_has = target_factor in pred_map.get(key, set())
        if g_has and p_has:
            tp += 1
        elif g_has and not p_has:
            fn += 1
    # FP: pred has factor but gold doesn't
    for key, pred_factors in pred_map.items():
        if target_factor in pred_factors and target_factor not in gold_map.get(key, set()):
            fp += 1
    
    f1 = f1_from_counts(tp, fp, fn)
    return f1, tp, fp, fn

print('Per-element evaluation functions loaded')

Per-element evaluation functions loaded


## Run Per-Element Evaluation

In [12]:
# ============================================================
# Run per-element evaluation across all conversations
# ============================================================
SUGGESTION_TYPES = ['Strong', 'Moderate', 'Weak']
RESPONSE_TYPES = ['Agreeable', 'Moderate', 'Disagreeable']
PERCEPTIONS = ['Positive', 'Neutral', 'Negative', 'Mix']
FACTORS = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7']

element_rows = []

for persona, dirs in [('Japanese', JAPANESE_DIRS), ('American', AMERICAN_DIRS)]:
    for model_label in dirs:
        result_dir = dirs[model_label]
        if not os.path.isdir(result_dir):
            continue

        logs = sorted([d for d in os.listdir(result_dir)
                       if d.endswith('_log') and os.path.isdir(os.path.join(result_dir, d))])

        for log_name in logs:
            gold = load_gold(log_name)
            if not gold:
                continue
            final = load_4step_final(result_dir, log_name)
            if not final:
                continue

            scope_p = set(final['step1']['participants'])
            scope_r = set(final['step1']['restaurant_brands'])

            gold_s12 = gold.get('step1_2', {})
            gold_s3 = gold.get('step3', {})
            gold_s4 = gold.get('step4', {})
            
            pred_suggestion = final['step1'].get('suggestion_table', [])
            pred_response = final['step1'].get('response_table', [])
            pred_sentiment = final['step3'].get('sentiment_table', [])
            pred_pref = final['step4']['f1'].get('preference_table', [])
            pred_cons = final['step4']['f1'].get('constraint_table', [])

            row = {'persona': persona, 'model': model_label, 'log': log_name}

            # Step 1.2: Suggestion per type
            for st in SUGGESTION_TYPES:
                f1, tp, fp, fn = step12_per_type_f1(
                    gold_s12.get('suggestion_table', []), pred_suggestion,
                    'suggestion_type', st)
                row[f'Suggestion_{st}_F1'] = f1
                row[f'Suggestion_{st}_support'] = tp + fn  # gold count

            # Step 1.2: Response per type
            for rt in RESPONSE_TYPES:
                f1, tp, fp, fn = step12_per_type_f1(
                    gold_s12.get('response_table', []), pred_response,
                    'response_type', rt)
                row[f'Response_{rt}_F1'] = f1
                row[f'Response_{rt}_support'] = tp + fn

            # Step 3: Perception per type (scope-filtered)
            gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
            for pv in PERCEPTIONS:
                f1, tp, fp, fn = step3_per_perception_f1(
                    gold_s3_table, pred_sentiment, pv, scope_p, scope_r)
                row[f'Perception_{pv}_F1'] = f1
                row[f'Perception_{pv}_support'] = tp + fn

            # Step 4: Factor-level positive-F1 (scope-filtered)
            for fac in FACTORS:
                f1, tp, fp, fn = step4_per_factor_positive_f1(
                    gold_s4.get('preference_table', []),
                    gold_s4.get('constraint_table', []),
                    pred_pref, pred_cons,
                    scope_p, scope_r, fac)
                row[f'Factor_{fac}_F1'] = f1
                row[f'Factor_{fac}_support'] = tp + fn

            element_rows.append(row)

df_elem = pd.DataFrame(element_rows)
print(f'Per-element evaluation: {len(df_elem)} rows')
print(f'Columns: {len(df_elem.columns)}')

Per-element evaluation: 281 rows
Columns: 37


### Step 1.2: Suggestion Type & Response Type F1

In [13]:
# Step 1.2 Suggestion per type
sugg_cols = [f'Suggestion_{st}_F1' for st in SUGGESTION_TYPES]
resp_cols = [f'Response_{rt}_F1' for rt in RESPONSE_TYPES]

print('=' * 120)
print('  Step 1.2: Suggestion Type F1 (Japanese vs American)')
print('=' * 120)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n  {model}')
    print(f'  {"Type":<15} | {"Japanese":>22} | {"American":>22} | {"Δ":>12} | {"Gold count":>10}')
    print(f'  {"-"*90}')
    for st in SUGGESTION_TYPES:
        col = f'Suggestion_{st}_F1'
        sup_col = f'Suggestion_{st}_support'
        jpn = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'Japanese')]
        amer = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'American')]
        # Only count conversations where this type exists in gold
        jpn_has = jpn[jpn[sup_col] > 0]
        amer_has = amer[amer[sup_col] > 0]
        jm = jpn_has[col].mean() if len(jpn_has) > 0 else float('nan')
        am = amer_has[col].mean() if len(amer_has) > 0 else float('nan')
        js = jpn_has[col].std() if len(jpn_has) > 0 else float('nan')
        a_s = amer_has[col].std() if len(amer_has) > 0 else float('nan')
        delta = am - jm
        total_gold = int(jpn[sup_col].sum())
        print(f'  {st:<15} | {jm:.4f} (±{js:.4f}) | {am:.4f} (±{a_s:.4f}) | {delta:>+.4f} | {total_gold:>10}')

print('\n\n' + '=' * 120)
print('  Step 1.2: Response Type F1 (Japanese vs American)')
print('=' * 120)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n  {model}')
    print(f'  {"Type":<15} | {"Japanese":>22} | {"American":>22} | {"Δ":>12} | {"Gold count":>10}')
    print(f'  {"-"*90}')
    for rt in RESPONSE_TYPES:
        col = f'Response_{rt}_F1'
        sup_col = f'Response_{rt}_support'
        jpn = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'Japanese')]
        amer = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'American')]
        jpn_has = jpn[jpn[sup_col] > 0]
        amer_has = amer[amer[sup_col] > 0]
        jm = jpn_has[col].mean() if len(jpn_has) > 0 else float('nan')
        am = amer_has[col].mean() if len(amer_has) > 0 else float('nan')
        js = jpn_has[col].std() if len(jpn_has) > 0 else float('nan')
        a_s = amer_has[col].std() if len(amer_has) > 0 else float('nan')
        delta = am - jm
        total_gold = int(jpn[sup_col].sum())
        print(f'  {rt:<15} | {jm:.4f} (±{js:.4f}) | {am:.4f} (±{a_s:.4f}) | {delta:>+.4f} | {total_gold:>10}')

  Step 1.2: Suggestion Type F1 (Japanese vs American)

  GPT-5
  Type            |               Japanese |               American |            Δ | Gold count
  ------------------------------------------------------------------------------------------
  Strong          | 0.8056 (±0.3882) | 0.8889 (±0.2959) | +0.0833 |         14
  Moderate        | 0.8865 (±0.1531) | 0.8830 (±0.1634) | -0.0035 |        137
  Weak            | 0.6451 (±0.4026) | 0.5392 (±0.4391) | -0.1059 |         26

  GPT-4o
  Type            |               Japanese |               American |            Δ | Gold count
  ------------------------------------------------------------------------------------------
  Strong          | 0.3889 (±0.4889) | 0.3056 (±0.4597) | -0.0833 |         14
  Moderate        | 0.8184 (±0.1550) | 0.8271 (±0.1556) | +0.0087 |        137
  Weak            | 0.6647 (±0.3576) | 0.7020 (±0.3645) | +0.0373 |         26

  Type            |               Japanese |               American |     

### Step 3: Perception F1 per Category

In [14]:
print('=' * 120)
print('  Step 3: Perception F1 per Category (Japanese vs American, scope-filtered)')
print('=' * 120)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n  {model}')
    print(f'  {"Perception":<15} | {"Japanese":>22} | {"American":>22} | {"Δ":>12} | {"Gold count":>10}')
    print(f'  {"-"*90}')
    for pv in PERCEPTIONS:
        col = f'Perception_{pv}_F1'
        sup_col = f'Perception_{pv}_support'
        jpn = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'Japanese')]
        amer = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'American')]
        jpn_has = jpn[jpn[sup_col] > 0]
        amer_has = amer[amer[sup_col] > 0]
        jm = jpn_has[col].mean() if len(jpn_has) > 0 else float('nan')
        am = amer_has[col].mean() if len(amer_has) > 0 else float('nan')
        js = jpn_has[col].std() if len(jpn_has) > 0 else float('nan')
        a_s = amer_has[col].std() if len(amer_has) > 0 else float('nan')
        delta = am - jm
        total_gold = int(jpn[sup_col].sum())
        print(f'  {pv:<15} | {jm:.4f} (±{js:.4f}) | {am:.4f} (±{a_s:.4f}) | {delta:>+.4f} | {total_gold:>10}')

  Step 3: Perception F1 per Category (Japanese vs American, scope-filtered)

  GPT-5
  Perception      |               Japanese |               American |            Δ | Gold count
  ------------------------------------------------------------------------------------------
  Positive        | 0.7515 (±0.1666) | 0.7515 (±0.1612) | +0.0000 |        281
  Neutral         | 0.8547 (±0.1222) | 0.8584 (±0.1248) | +0.0038 |        803
  Negative        | 0.5016 (±0.4075) | 0.4619 (±0.3937) | -0.0397 |         82
  Mix             | 0.4223 (±0.4119) | 0.4422 (±0.3765) | +0.0199 |         39

  GPT-4o
  Perception      |               Japanese |               American |            Δ | Gold count
  ------------------------------------------------------------------------------------------
  Positive        | 0.6532 (±0.2125) | 0.6904 (±0.2190) | +0.0372 |        281
  Neutral         | 0.7744 (±0.2285) | 0.7965 (±0.2109) | +0.0221 |        788
  Negative        | 0.3935 (±0.3908) | 0.3655 (±0.386

### Step 4: Factor-Level Positive-F1

In [15]:
print('=' * 120)
print('  Step 4: Factor-Level Positive-F1 (Japanese vs American, scope-filtered)')
print('=' * 120)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n  {model}')
    print(f'  {"Factor":<10} | {"Japanese":>22} | {"American":>22} | {"Δ":>12} | {"Gold count":>10}')
    print(f'  {"-"*85}')
    for fac in FACTORS:
        col = f'Factor_{fac}_F1'
        sup_col = f'Factor_{fac}_support'
        jpn = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'Japanese')]
        amer = df_elem[(df_elem['model'] == model) & (df_elem['persona'] == 'American')]
        jpn_has = jpn[jpn[sup_col] > 0]
        amer_has = amer[amer[sup_col] > 0]
        jm = jpn_has[col].mean() if len(jpn_has) > 0 else float('nan')
        am = amer_has[col].mean() if len(amer_has) > 0 else float('nan')
        js = jpn_has[col].std() if len(jpn_has) > 0 else float('nan')
        a_s = amer_has[col].std() if len(amer_has) > 0 else float('nan')
        delta = am - jm
        total_gold = int(jpn[sup_col].sum())
        print(f'  {fac:<10} | {jm:.4f} (±{js:.4f}) | {am:.4f} (±{a_s:.4f}) | {delta:>+.4f} | {total_gold:>10}')

  Step 4: Factor-Level Positive-F1 (Japanese vs American, scope-filtered)

  GPT-5
  Factor     |               Japanese |               American |            Δ | Gold count
  -------------------------------------------------------------------------------------
  A1         | 0.5193 (±0.2504) | 0.5545 (±0.2482) | +0.0352 |        163
  A2         | 0.5482 (±0.2797) | 0.5507 (±0.3151) | +0.0026 |         32
  A3         | 0.8071 (±0.2694) | 0.8250 (±0.2363) | +0.0179 |          8
  A4         | 0.4429 (±0.3377) | 0.3500 (±0.2742) | -0.0929 |          7
  A5         | 0.5449 (±0.2327) | 0.4362 (±0.2578) | -0.1087 |         22
  A6         | 0.6438 (±0.2830) | 0.6512 (±0.2898) | +0.0074 |         40
  A7         | 0.1064 (±0.2294) | 0.1393 (±0.2683) | +0.0329 |        185

  GPT-4o
  Factor     |               Japanese |               American |            Δ | Gold count
  -------------------------------------------------------------------------------------
  A1         | 0.4858 (±0.2182)

### Save Per-Element Results

In [16]:
df_elem.to_csv('appendix/results/american/american_eval_per_element.csv', index=False)

# Also create a compact summary
elem_summary_rows = []
for persona in ['Japanese', 'American']:
    for model in ['GPT-5', 'GPT-4o']:
        sub = df_elem[(df_elem['persona'] == persona) & (df_elem['model'] == model)]
        row = {'persona': persona, 'model': model}
        for st in SUGGESTION_TYPES:
            has = sub[sub[f'Suggestion_{st}_support'] > 0]
            row[f'Suggestion_{st}'] = has[f'Suggestion_{st}_F1'].mean() if len(has) > 0 else None
        for rt in RESPONSE_TYPES:
            has = sub[sub[f'Response_{rt}_support'] > 0]
            row[f'Response_{rt}'] = has[f'Response_{rt}_F1'].mean() if len(has) > 0 else None
        for pv in PERCEPTIONS:
            has = sub[sub[f'Perception_{pv}_support'] > 0]
            row[f'Perception_{pv}'] = has[f'Perception_{pv}_F1'].mean() if len(has) > 0 else None
        for fac in FACTORS:
            has = sub[sub[f'Factor_{fac}_support'] > 0]
            row[f'Factor_{fac}'] = has[f'Factor_{fac}_F1'].mean() if len(has) > 0 else None
        elem_summary_rows.append(row)

df_elem_summary = pd.DataFrame(elem_summary_rows)
df_elem_summary.to_csv('appendix/results/american/american_eval_per_element_summary.csv', index=False)

print('Saved:')
print('  - appendix/results/american/american_eval_per_element.csv (per conversation detail)')
print('  - appendix/results/american/american_eval_per_element_summary.csv (compact summary)')
display(df_elem_summary.set_index(['persona', 'model']).T)

Saved:
  - american_eval_per_element.csv (per conversation detail)
  - american_eval_per_element_summary.csv (compact summary)


persona                Japanese                      American            \
Suggestion_Strong      0.805556  0.388889  0.722222  0.888889  0.305556   
Suggestion_Moderate    0.886473  0.818377  0.820635  0.882998  0.827055   
Suggestion_Weak        0.645098  0.664706  0.694118  0.539216  0.701961   
Response_Agreeable     0.771429  0.714890  0.765684  0.785714  0.711716   
Response_Moderate      0.538265  0.159410  0.478345  0.554535  0.191043   
Response_Disagreeable  0.277778  0.111111  0.111111  0.277778  0.111111   
Perception_Positive    0.751484  0.653202  0.726950  0.751515  0.690415   
Perception_Neutral     0.854672  0.774439  0.854305  0.858433  0.796537   
Perception_Negative    0.501563  0.393519  0.396230  0.461883  0.365542   
Perception_Mix         0.422294  0.121212  0.150000  0.442218  0.075758   
Factor_A1              0.519306  0.485767  0.536316  0.554550  0.496341   
Factor_A2              0.548153  0.340115  0.350794  0.550741  0.454921   
Factor_A3              0.807143  0.600000  0.593750  0.825000  0.475000   
Factor_A4              0.442857  0.083333  0.083333  0.350000  0.125000   
Factor_A5              0.544868  0.336190  0.490476  0.436190  0.348254   
Factor_A6              0.643758  0.609159  0.623183  0.651179  0.611571   
Factor_A7              0.106386  0.013158  0.018045  0.139296  0.000000   

persona                          
Suggestion_Strong      0.606061  
Suggestion_Moderate    0.856530  
Suggestion_Weak        0.792157  
Response_Agreeable     0.762509  
Response_Moderate      0.407724  
Response_Disagreeable  0.333333  
Perception_Positive    0.697612  
Perception_Neutral     0.840299  
Perception_Negative    0.382021  
Perception_Mix         0.045455  
Factor_A1              0.499990  
Factor_A2              0.288889  
Factor_A3              0.615385  
Factor_A4              0.154762  
Factor_A5              0.237755  
Factor_A6              0.589181  
Factor_A7              0.033033

## Confusion Matrix Heatmaps
Generate confusion matrix PNGs for each model × persona combination:
- Step 1.2: Suggestion (Strong/Moderate/Weak), Response (Agreeable/Moderate/Disagreeable)
- Step 3: Sentiment/Perception (Positive/Neutral/Negative/Mix)
- Step 4: Integrated Table factors (a1–a7 + none)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import defaultdict

# ============================================================
# Heatmap plotting function (same style as original)
# ============================================================
def plot_crosstab_heatmap(crosstab_df, title, filepath):
    rows, cols = crosstab_df.shape
    if rows == 0 or cols == 0:
        print(f"  SKIP: '{title}' - no data")
        return

    cell_size = 0.9
    figsize_w = max(5, cols * cell_size + 2)
    figsize_h = max(4, rows * cell_size + 2)

    fig, ax = plt.subplots(figsize=(figsize_w, figsize_h))
    sns.heatmap(
        crosstab_df,
        annot=True, fmt='d', cmap='Blues',
        linewidths=0.5, linecolor='lightgray',
        annot_kws={"size": 11, "weight": "bold"},
        square=True, cbar=True, ax=ax
    )
    ax.set_title(title, fontsize=14, pad=15, fontweight='bold')
    ax.set_xlabel('Predicted', fontweight='bold', fontsize=12)
    ax.set_ylabel('Answer', fontweight='bold', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11, fontweight='bold')
    plt.yticks(rotation=0, fontsize=11, fontweight='bold')
    plt.tight_layout(pad=1.2)
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {filepath}")

# ============================================================
# Build confusion matrices from final_results.json + gold
# ============================================================
def build_all_confusion_matrices(result_dir, gold_dir, persona_label, model_label, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    prefix = f"{persona_label}_{model_label}"

    logs = sorted([d for d in os.listdir(result_dir)
                   if d.endswith('_log') and os.path.isdir(os.path.join(result_dir, d))])

    # Accumulators
    sugg_pairs = []   # (gold_type, pred_type)
    resp_pairs = []
    sent_pairs = []   # (gold_perception, pred_sentiment)
    factor_pairs = [] # (gold_factor, pred_factor)

    for log_name in logs:
        # Load gold
        gold = {}
        for key, fname in [('step1_2', 'step1_2_gold.json'), ('step3', 'step3_gold.json'), ('step4', 'step4_gold.json')]:
            fpath = os.path.join(gold_dir, log_name, fname)
            if os.path.exists(fpath):
                with open(fpath, 'r', encoding='utf-8') as f:
                    gold[key] = json.load(f)

        # Load prediction
        final_path = os.path.join(result_dir, log_name, f'{log_name}_final_results.json')
        if not os.path.exists(final_path):
            continue
        with open(final_path, 'r', encoding='utf-8') as f:
            final = json.load(f)

        scope_p = set(final['step1']['participants'])
        scope_r = {normalize_name(r) for r in final['step1']['restaurant_brands']}

        # --- Step 1.2: Suggestion ---
        gold_s12 = gold.get('step1_2', {})
        gold_sugg = {r['participant']: r['suggestion_type'] for r in gold_s12.get('suggestion_table', [])}
        pred_sugg = {r['participant']: r.get('suggestion_type', '') for r in final['step1'].get('suggestion_table', [])}
        for p in set(list(gold_sugg.keys()) + list(pred_sugg.keys())):
            g = gold_sugg.get(p, '')
            pr = pred_sugg.get(p, '')
            if g:  # only if gold has a value
                sugg_pairs.append((g.lower(), pr.lower() if pr else 'missing'))

        # --- Step 1.2: Response ---
        gold_resp = {r['participant']: r['response_type'] for r in gold_s12.get('response_table', [])}
        pred_resp = {r['participant']: r.get('response_type', '') for r in final['step1'].get('response_table', [])}
        for p in set(list(gold_resp.keys()) + list(pred_resp.keys())):
            g = gold_resp.get(p, '')
            pr = pred_resp.get(p, '')
            if g:
                resp_pairs.append((g.lower(), pr.lower() if pr else 'missing'))

        # --- Step 3: Perception/Sentiment (scope-filtered) ---
        gold_s3 = gold.get('step3', {})
        gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
        pred_s3_table = final['step3'].get('sentiment_table', final['step3'].get('perception_table', []))

        gold_s3_map = {}
        for row in gold_s3_table:
            p, r = row.get('participant', ''), normalize_name(row.get('restaurant', ''))
            if p in scope_p and r in scope_r:
                gold_s3_map[(p, r)] = row.get('perception', row.get('sentiment', '')).strip()

        pred_s3_map = {}
        for row in pred_s3_table:
            p, r = row.get('participant', ''), normalize_name(row.get('restaurant', ''))
            if p in scope_p and r in scope_r:
                pred_s3_map[(p, r)] = row.get('sentiment', row.get('perception', '')).strip()

        for key, gv in gold_s3_map.items():
            pv = pred_s3_map.get(key, 'Missing')
            if gv:
                sent_pairs.append((gv, pv if pv else 'Missing'))

        # --- Step 4: Integrated Table (scope-filtered) ---
        gold_s4 = gold.get('step4', {})
        pred_pref = final['step4']['f1'].get('preference_table', [])
        pred_cons = final['step4']['f1'].get('constraint_table', [])

        def build_integrated_map(pref_tbl, cons_tbl, pref_key='preferences', cons_key='constraints'):
            m = {}
            for row in pref_tbl:
                p, r = row['participant'], normalize_name(row['restaurant'])
                if p in scope_p and r in scope_r:
                    m.setdefault((p, r), set()).update(parse_factors(row.get(pref_key, 'None')))
            for row in cons_tbl:
                p, r = row['participant'], normalize_name(row['restaurant'])
                if p in scope_p and r in scope_r:
                    m.setdefault((p, r), set()).update(parse_factors(row.get(cons_key, 'None')))
            return m

        gold_int = build_integrated_map(
            gold_s4.get('preference_table', []), gold_s4.get('constraint_table', []))
        pred_int = build_integrated_map(pred_pref, pred_cons)

        all_factors = {'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7'}
        for key, gold_factors in gold_int.items():
            pred_factors = pred_int.get(key, set())
            if not gold_factors:  # skip None/empty gold cells
                continue
            for gf in gold_factors:
                if gf.lower() not in all_factors:
                    continue
                if pred_factors:
                    for pf in pred_factors:
                        factor_pairs.append((gf.lower(), pf.lower()))
                else:
                    factor_pairs.append((gf.lower(), 'none'))

    # --- Build and plot crosstabs ---
    # Step 1.2 Suggestion
    if sugg_pairs:
        labels = ['moderate', 'strong', 'weak']
        df_s = pd.DataFrame(sugg_pairs, columns=['Answer', 'Predicted'])
        ct = pd.crosstab(df_s['Answer'], df_s['Predicted'])
        # Reindex to standard labels
        all_labels = sorted(set(labels) | set(ct.index) | set(ct.columns))
        ct = ct.reindex(index=[l for l in labels if l in ct.index],
                       columns=[l for l in all_labels if l in ct.columns], fill_value=0)
        plot_crosstab_heatmap(ct, f'Step1.2 Suggestion ({prefix})',
                             os.path.join(output_dir, f'{prefix}_Step1_2_Suggestion_CM.png'))

    # Step 1.2 Response
    if resp_pairs:
        labels = ['agreeable', 'disagreeable', 'moderate']
        df_r = pd.DataFrame(resp_pairs, columns=['Answer', 'Predicted'])
        ct = pd.crosstab(df_r['Answer'], df_r['Predicted'])
        all_labels = sorted(set(labels) | set(ct.index) | set(ct.columns))
        ct = ct.reindex(index=[l for l in labels if l in ct.index],
                       columns=[l for l in all_labels if l in ct.columns], fill_value=0)
        plot_crosstab_heatmap(ct, f'Step1.2 Response ({prefix})',
                             os.path.join(output_dir, f'{prefix}_Step1_2_Response_CM.png'))

    # Step 3 Sentiment
    if sent_pairs:
        labels = ['Mix', 'Negative', 'Neutral', 'Positive']
        df_p = pd.DataFrame(sent_pairs, columns=['Answer', 'Predicted'])
        ct = pd.crosstab(df_p['Answer'], df_p['Predicted'])
        all_labels_r = [l for l in labels if l in ct.index]
        all_labels_c = sorted(set(ct.columns))
        ct = ct.reindex(index=all_labels_r, columns=all_labels_c, fill_value=0)
        plot_crosstab_heatmap(ct, f'Step3. Sentiment Table ({prefix})',
                             os.path.join(output_dir, f'{prefix}_Step3_Perception_CM.png'))

    # Step 4 Integrated
    if factor_pairs:
        factor_labels = ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7']
        df_f = pd.DataFrame(factor_pairs, columns=['Answer', 'Predicted'])
        ct = pd.crosstab(df_f['Answer'], df_f['Predicted'])
        all_cols = sorted(set(factor_labels + ['none']) & set(ct.columns))
        # Ensure 'none' is last column
        cols_ordered = [c for c in factor_labels if c in ct.columns] + (['none'] if 'none' in ct.columns else [])
        # Add any other unexpected columns
        for c in ct.columns:
            if c not in cols_ordered:
                cols_ordered.append(c)
        rows_ordered = [r for r in factor_labels if r in ct.index]
        ct = ct.reindex(index=rows_ordered, columns=cols_ordered, fill_value=0)
        plot_crosstab_heatmap(ct, f'Step4. Integrated Table ({prefix})',
                             os.path.join(output_dir, f'{prefix}_Step4_Integrated_CM.png'))

    print(f"  Done: {prefix}")
    return len(sugg_pairs), len(resp_pairs), len(sent_pairs), len(factor_pairs)

print('Confusion matrix functions loaded')

## Generate All Confusion Matrix PNGs

In [ ]:
# Generate confusion matrices for both personas × 2 models (GPT-4o temp=0, GPT-5)
CONFIGS = {
    'Japanese': {
        'GPT-5':     'results/gpt5/raw',
        'GPT-4o': 'results/gpt4o/raw',
    },
    'American': {
        'GPT-5':     'appendix/results/american/gpt5/raw',
        'GPT-4o': 'appendix/results/american/gpt4o/raw',
    }
}

CM_OUTPUT_DIR = 'appendix/results/confusion_matrix'
os.makedirs(CM_OUTPUT_DIR, exist_ok=True)

for persona, models in CONFIGS.items():
    for model_label, result_dir in models.items():
        if not os.path.isdir(result_dir):
            print(f"SKIP: {persona}/{model_label} - {result_dir} not found")
            continue
        print(f"\n{'='*60}")
        print(f"  {persona} / {model_label}")
        print(f"{'='*60}")
        counts = build_all_confusion_matrices(
            result_dir, GOLD_DIR, persona, model_label, CM_OUTPUT_DIR
        )
        print(f"  Pairs: suggestion={counts[0]}, response={counts[1]}, sentiment={counts[2]}, factors={counts[3]}")

# List generated files
print(f"\n{'='*60}")
print(f"Generated PNG files in {CM_OUTPUT_DIR}/:")
for f in sorted(os.listdir(CM_OUTPUT_DIR)):
    if f.endswith('.png'):
        print(f"  {f}")